In [33]:
import os
import yfinance as yf
import pandas as pd

# 1. Define Constants for where data will live
STOCK_PATH = os.path.join("datasets", "stocks")

def fetch_stock_data(ticker, start_date="2023-01-01", end_date=None, stock_path=STOCK_PATH):
    """
    Downloads stock data from Yahoo Finance and saves it as a CSV file.
    """
    # Create the directory if it doesn't exist
    if not os.path.isdir(stock_path):
        os.makedirs(stock_path)
    
    # Construct the file path
    csv_path = os.path.join(stock_path, f"{ticker}.csv")
    
    # Download data using yfinance
    print(f"Downloading data for {ticker}...")
    data = yf.download(ticker, start=start_date, end=end_date, progress=False)
    
    # Save to CSV
    if not data.empty:
        data.to_csv(csv_path)
        print(f"Saved {ticker} data to {csv_path}")
    else:
        print(f"No data found for {ticker}")



In [34]:
def load_stock_data(ticker, stock_path=STOCK_PATH):
    """
    Loads the locally saved stock data CSV into a Pandas DataFrame.
    """
    csv_path = os.path.join(stock_path, f"{ticker}.csv")
    
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No data found at {csv_path}.")
    
    # FIX: 
    # header=0    -> Use the first line (Price, Close, High...) as column names
    # skiprows=[1,2] -> Ignore the 'Ticker' and empty 'Date' rows
    # index_col=0 -> The first column is the Index
    df = pd.read_csv(csv_path, header=0, skiprows=[1, 2], index_col=0, parse_dates=True)
    
    # Optional: Rename the index from "Price" to "Date" for clarity
    df.index.name = "Date"
    
    return df

In [35]:
fetch_stock_data("NVDA", start_date="2023-01-01")

C:\Users\dilip\AppData\Local\Temp\ipykernel_8212\982259937.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)


Saved NVDA data to datasets\stocks\NVDA.csv


In [36]:
df = load_stock_data("AAPL")
print(df.head())

                 Close        High         Low        Open     Volume
Date                                                                 
2023-01-03  123.211189  128.954537  122.324564  128.343757  112117500
2023-01-04  124.482056  126.747876  123.221080  125.004178   89113600
2023-01-05  123.161949  125.871079  122.905819  125.240591   80962700
2023-01-06  127.693596  128.353637  123.033897  124.137254   87754700
2023-01-09  128.215698  131.427258  127.959568  128.530950   70790800


In [37]:
import numpy as np
# --- NEW FUNCTION IMPLEMENTING YOUR FORMULA ---
def calculate_volatility(df, M, dt=1/252):
    """
    Implements the annualized volatility formula:
    sigma = sqrt( (1 / ((M-1)*dt)) * sum( (R_i - R_bar)^2 ) )
    
    Parameters:
    df (pd.DataFrame): DataFrame containing 'Close' prices.
    M (int): The window size (number of observations).
    dt (float): Time step in years (default 1/252 for daily data).
    
    Returns:
    pd.Series: Rolling volatility values.
    """
    
    # 1. Calculate Log Returns: R_i = ln(Price_t / Price_{t-1})
    # We use 'Close' or 'Adj Close'
    #df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
    df['Log_Return'] = df['Close'].pct_change()
    
    # 2. Define the rolling window function
    # The formula is essentially: standard_deviation / sqrt(dt)
    # But let's implement it exactly as written algebraically for clarity.
    
    def formula_logic(window_data):
        # window_data represents the vector of R_i values of length M
        if len(window_data) < M: return np.nan
        
        R_bar = np.mean(window_data)
        sum_sq_diff = np.sum((window_data - R_bar)**2)
        
        # The term inside the square root
        variance_term = (1 / ((M - 1) * dt)) * sum_sq_diff
        
        return np.sqrt(variance_term)

    # 3. Apply rolling window
    # .rolling(M) creates windows of size M
    volatility = df['Log_Return'].rolling(window=M).apply(formula_logic, raw=True)
    
    return volatility

In [38]:
M_parameter = 20 # Example: 21 days (approx 1 trading month)
delta_t = 1/252  # Daily data
# Calculate
df['Volatility_M'] = calculate_volatility(df, M=M_parameter, dt=delta_t)

print(f"Data with Volatility (M={M_parameter}):")
print(df[['Close', 'Log_Return', 'Volatility_M']].tail())

Data with Volatility (M=20):
                 Close  Log_Return  Volatility_M
Date                                            
2025-12-08  277.890015   -0.003192      0.170527
2025-12-09  277.179993   -0.002555      0.170888
2025-12-10  278.779999    0.005772      0.154624
2025-12-11  278.029999   -0.002690      0.152879
2025-12-12  278.279999    0.000899      0.152529


In [39]:
import numpy as np
# --- NEW FUNCTION IMPLEMENTING YOUR FORMULA ---
def calculate_volatilitylog(df, M, dt=1/252):
    """
    Implements the annualized volatility formula:
    sigma = sqrt( (1 / ((M-1)*dt)) * sum( (R_i - R_bar)^2 ) )
    
    Parameters:
    df (pd.DataFrame): DataFrame containing 'Close' prices.
    M (int): The window size (number of observations).
    dt (float): Time step in years (default 1/252 for daily data).
    
    Returns:
    pd.Series: Rolling volatility values.
    """
    
    # 1. Calculate Log Returns: R_i = ln(Price_t / Price_{t-1})
    # We use 'Close' or 'Adj Close'
    df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

    
    # 2. Define the rolling window function
    # The formula is essentially: standard_deviation / sqrt(dt)
    # But let's implement it exactly as written algebraically for clarity.
    
    def formula_logic(window_data):
        # window_data represents the vector of R_i values of length M
        if len(window_data) < M: return np.nan
        
        R_bar = np.mean(window_data)
        sum_sq_diff = np.sum((window_data - R_bar)**2)
        
        # The term inside the square root
        variance_term = (1 / ((M - 1) * dt)) * sum_sq_diff
        
        return np.sqrt(variance_term)

    # 3. Apply rolling window
    # .rolling(M) creates windows of size M
    volatility = df['Log_Return'].rolling(window=M).apply(formula_logic, raw=True)
    
    return volatility

In [40]:
M_parameterlog = 20 # Example: 21 days (approx 1 trading month)
delta_tlog = 1/252  # Daily data
# Calculate
df['Volatility_Mlog'] = calculate_volatilitylog(df, M=M_parameter, dt=delta_t)

print(f"Data with Volatility (M={M_parameter}):")
print(df[['Close', 'Log_Return', 'Volatility_Mlog']].tail())

Data with Volatility (M=20):
                 Close  Log_Return  Volatility_Mlog
Date                                               
2025-12-08  277.890015   -0.003198         0.170014
2025-12-09  277.179993   -0.002558         0.170349
2025-12-10  278.779999    0.005756         0.154357
2025-12-11  278.029999   -0.002694         0.152612
2025-12-12  278.279999    0.000899         0.152271
